# Feature Engineering

TODO: purpose of this notebook.

## Read Dataset

In [74]:
import pandas as pd
import numpy as np

cleaned_path = r"D:\NUS Note\IT5006\it5006_project\data\preprocessed_data\delivery_time_regression.csv"   # 修改成你的实际路径

df = pd.read_csv(cleaned_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


Dataset shape: (96454, 52)

Columns:
['order_id', 'order_purchase_timestamp', 'purchase_month', 'customer_state', 'customer_region', 'customer_zip_code_prefix', 'customer_zip_lat', 'customer_zip_lng', 'customer_zip_density', 'customer_is_capital', 'remote_flag', 'order_value', 'mean_item_price', 'max_item_price', 'freight_value', 'item_count', 'unique_product_count', 'seller_count', 'seller_state_count', 'product_category_count', 'multi_seller_flag', 'product_category', 'primary_seller_state', 'primary_seller_region', 'region_pair', 'same_state', 'any_seller_same_state', 'same_city', 'any_seller_same_city', 'mean_seller_zip_lat', 'mean_seller_zip_lng', 'estimated_min_distance_km', 'estimated_mean_distance_km', 'estimated_max_distance_km', 'geolocation_missing', 'total_weight_g', 'total_volume_cm3', 'max_item_weight_g', 'longest_side_cm', 'max_sum_of_dims_cm', 'volumetric_weight_kg', 'billable_weight_kg', 'density_g_cm3', 'payment_value', 'payment_installments', 'payment_type_count', 'p

## 1. Product features (volume, weight, category)

In [75]:
# TODO

## 2. Geography features (customer/seller state, same-state flag)

In [76]:
# TODO

## 3. Time features (purchase year, month, weekday)

In [77]:
# TODO
# Convert purchase timestamp
df["order_purchase_timestamp"] = pd.to_datetime(
    df["order_purchase_timestamp"],
    errors="coerce"
)

print("\nMissing purchase timestamps:",
      df["order_purchase_timestamp"].isna().sum())

# Year
df["purchase_year"] = (
    df["order_purchase_timestamp"].dt.year
)

# Month
# Recalculate from timestamp to ensure consistency
df["purchase_month"] = (
    df["order_purchase_timestamp"].dt.month
)

# Day of week
# Monday = 0, Sunday = 6
df["purchase_weekday"] = (
    df["order_purchase_timestamp"].dt.weekday
)

# Hour of day
df["purchase_hour"] = (
    df["order_purchase_timestamp"].dt.hour
)

# Weekend indicator
# Saturday / Sunday = 1
df["purchase_is_weekend"] = (
    df["purchase_weekday"] >= 5
).astype(int)


Missing purchase timestamps: 0


## 4. Payment Features

In [78]:
# Payment per installment
# Replace 0 installments with NaN temporarily
# to avoid division by zero
installments_safe = (
    df["payment_installments"]
    .replace(0, np.nan)
)

df["payment_per_installment"] = (
    df["payment_value"] / installments_safe
)

#Installment flag
# 0 = no installment / one payment
# 1 = installment payment
df["installment_flag"] = (
    df["payment_installments"] > 1
).astype(int)

# Multiple payment types flag
df["multiple_payment_types"] = (
    df["payment_type_count"] > 1
).astype(int)

In [79]:
# Check category distributions

new_features = [
    "purchase_year",
    "purchase_month",
    "purchase_weekday",
    "purchase_hour",
    "purchase_is_weekend",
    "payment_per_installment",
    "installment_flag",
    "multiple_payment_types"
]

print("\n===== New Feature Missingness =====")

print(
    df[new_features]
    .isna()
    .sum()
)

print("\n===== Purchase Year =====")
print(
    df["purchase_year"]
    .value_counts()
    .sort_index()
)

print("\n===== Purchase Month =====")
print(
    df["purchase_month"]
    .value_counts()
    .sort_index()
)

print("\n===== Payment Type =====")
print(
    df["payment_type"]
    .value_counts(dropna=False)
)

print("\n===== Installment Flag =====")
print(
    df["installment_flag"]
    .value_counts(dropna=False)
)

print("\n===== Multiple Payment Types =====")
print(
    df["multiple_payment_types"]
    .value_counts(dropna=False)
)




===== New Feature Missingness =====
purchase_year              0
purchase_month             0
purchase_weekday           0
purchase_hour              0
purchase_is_weekend        0
payment_per_installment    3
installment_flag           0
multiple_payment_types     0
dtype: int64

===== Purchase Year =====
purchase_year
2016      267
2017    43410
2018    52777
Name: count, dtype: int64

===== Purchase Month =====
purchase_month
1      7818
2      8208
3      9546
4      9095
5     10294
6      9229
7     10026
8     10542
9      4151
10     4743
11     7288
12     5514
Name: count, dtype: int64

===== Payment Type =====
payment_type
credit_card    73202
boleto         19187
voucher         2581
debit_card      1483
Unknown            1
Name: count, dtype: int64

===== Installment Flag =====
installment_flag
1    49648
0    46806
Name: count, dtype: int64

===== Multiple Payment Types =====
multiple_payment_types
0    94273
1     2181
Name: count, dtype: int64


## 5. Seller Performance

In [80]:
# Load data
bridge_path = r"D:\NUS Note\IT5006\it5006_project\data\preprocessed_data\order_seller_bridge.csv"
order_seller_bridge = pd.read_csv(bridge_path)

# Convert timestamp
order_seller_bridge["order_purchase_timestamp"] = pd.to_datetime(
    order_seller_bridge["order_purchase_timestamp"],
    errors="coerce"
)

print("Cleaned dataset:", df.shape)
print("Order-seller bridge:", order_seller_bridge.shape)

Cleaned dataset: (96454, 59)
Order-seller bridge: (97795, 13)


In [81]:
# check bridge dataset id
duplicate_pairs = (
    order_seller_bridge
    .duplicated(
        subset=["order_id", "seller_id"],
        keep=False
    )
)

print(
    "Duplicated order-seller pairs:",
    duplicate_pairs.sum()
)

seller_count_check = (
    order_seller_bridge
    .groupby("order_id")["seller_id"]
    .nunique()
)

print(
    "Multi-seller orders:",
    (seller_count_check > 1).sum()
)

Duplicated order-seller pairs: 0
Multi-seller orders: 1275


In [82]:
# Merge outcome
seller_orders = order_seller_bridge.merge(
    df[
        [
            "order_id",
            "delivery_days",
            "promised_delivery_days"
        ]
    ],
    on="order_id",
    how="left",
    validate="many_to_one"
)

check_cols = [
    "seller_id",
    "order_purchase_timestamp",
    "split",
    "delivery_days",
    "promised_delivery_days"
]

print(
    seller_orders[check_cols]
    .isna()
    .sum()
)

seller_id                   0
order_purchase_timestamp    0
split                       0
delivery_days               0
promised_delivery_days      0
dtype: int64


In [83]:
# establish late_flag
# actual delivery_days > promised_delivery_days
seller_orders["late_flag"] = np.where(
    seller_orders["delivery_days"].notna()
    & seller_orders["promised_delivery_days"].notna(),
    
    (
        seller_orders["delivery_days"]
        >
        seller_orders["promised_delivery_days"]
    ).astype(int),
    
    np.nan
)

print(
    "Late flag distribution:",
    seller_orders["late_flag"].value_counts(dropna=False)
)

Late flag distribution: late_flag
0.0    90472
1.0     7323
Name: count, dtype: int64


In [84]:
train_so = seller_orders[
    seller_orders["split"] == "train"
].copy()

val_so = seller_orders[
    seller_orders["split"].isin(["validation", "val"])
].copy()

test_so = seller_orders[
    seller_orders["split"] == "test"
].copy()

# 先把同一个 Seller 的订单放在一起，再按照下单时间从早到晚排列
train_so = train_so.sort_values(
    [
        "seller_id",
        "order_purchase_timestamp",
        "order_id"
    ]
).reset_index(drop=True)

In [85]:
# Seller history order count
train_so["seller_hist_order_count"] = (
    train_so
    .groupby("seller_id")
    .cumcount()
)

# Average delivery days per seller
# 在当前订单之前，历史订单平均用了多少天配送
train_so["seller_hist_mean_delivery_days"] = (
    train_so
    .groupby("seller_id")["delivery_days"]
    .transform(
        lambda x: x.shift(1).expanding().mean()
    )
)

# delivery days variance per seller
train_so["seller_hist_std_delivery_days"] = (
    train_so
    .groupby("seller_id")["delivery_days"]
    .transform(
        lambda x: x.shift(1).expanding().std()
    )
)

# history late rate per seller
train_so["seller_hist_late_rate"] = (
    train_so
    .groupby("seller_id")["late_flag"]
    .transform(
        lambda x: x.shift(1).expanding().mean()
    )
)

# history item count per seller
train_so["seller_hist_item_count"] = (
    train_so
    .groupby("seller_id")["seller_item_count"]
    .transform(
        lambda x: x.shift(1).fillna(0).cumsum()
    )
)

# history order value per seller
train_so["seller_hist_order_value"] = (
    train_so
    .groupby("seller_id")["seller_order_value"]
    .transform(
        lambda x: x.shift(1).fillna(0).cumsum()
    )
)

# new seller flag
train_so["new_seller_flag"] = (
    train_so["seller_hist_order_count"] == 0
).astype(int)

In [86]:
# Global Fallback：新 Seller 完全没历史，用训练集总体平均
global_mean_delivery = (
    train_so["delivery_days"].mean()
)

global_std_delivery = (
    train_so["delivery_days"].std()
)

global_late_rate = (
    train_so["late_flag"].mean()
)

print("Global mean delivery:", global_mean_delivery)
print("Global delivery std:", global_std_delivery)
print("Global late rate:", global_late_rate)

Global mean delivery: 13.139496634038087
Global delivery std: 9.833532012793308
Global late rate: 0.07665632200122255


In [87]:
# Smoothed Mean: Seller自己的历史均值 + 平台整体均值
# SMOOTHING_ALPHA can be tuned to control the influence of global mean vs. seller-specific mean. 
# A higher alpha means more weight on the global mean, while a lower alpha gives more weight to the seller's own history.
SMOOTHING_ALPHA = 10

train_so["seller_hist_delivery_smoothed"] = (
    (
        train_so["seller_hist_order_count"]
        * train_so["seller_hist_mean_delivery_days"]
    )
    +
    (
        SMOOTHING_ALPHA
        * global_mean_delivery
    )
) / (
    train_so["seller_hist_order_count"]
    + SMOOTHING_ALPHA
)

train_so["seller_hist_delivery_smoothed"] = (
    train_so["seller_hist_delivery_smoothed"]
    .fillna(global_mean_delivery)
)

train_so["seller_hist_std_delivery_days"] = (
    train_so["seller_hist_std_delivery_days"]
    .fillna(global_std_delivery)
)

train_so["seller_hist_late_rate"] = (
    train_so["seller_hist_late_rate"]
    .fillna(global_late_rate)
)

In [88]:
# 查看 train_so 所有列名
print(train_so.columns.tolist())

['order_id', 'seller_id', 'order_purchase_timestamp', 'split', 'seller_city', 'seller_state', 'seller_zip_code_prefix', 'seller_zip_lat', 'seller_zip_lng', 'seller_item_count', 'seller_order_value', 'seller_freight_value', 'estimated_seller_customer_distance_km', 'delivery_days', 'promised_delivery_days', 'late_flag', 'seller_hist_order_count', 'seller_hist_mean_delivery_days', 'seller_hist_std_delivery_days', 'seller_hist_late_rate', 'seller_hist_item_count', 'seller_hist_order_value', 'new_seller_flag', 'seller_hist_delivery_smoothed']


In [89]:
# establish seller profile features for validation and test sets
seller_train_profile = (
    train_so
    .groupby("seller_id")
    .agg(
        seller_train_order_count=("order_id", "nunique"),
        seller_train_item_count=("seller_item_count", "sum"),
        seller_train_order_value=("seller_order_value", "sum"),
        seller_train_mean_delivery_days=("delivery_days", "mean"),
        seller_train_std_delivery_days=("delivery_days", "std"),
        seller_train_late_rate=("late_flag", "mean")
    )
    .reset_index()
)

# train profile smoothing
seller_train_profile["seller_train_delivery_smoothed"] = (
    (
        seller_train_profile["seller_train_order_count"]
        * seller_train_profile["seller_train_mean_delivery_days"]
    )
    +
    (
        SMOOTHING_ALPHA * global_mean_delivery
    )
) / (
    seller_train_profile["seller_train_order_count"]
    + SMOOTHING_ALPHA
)

seller_train_profile["seller_train_std_delivery_days"] = (
    seller_train_profile["seller_train_std_delivery_days"]
    .fillna(global_std_delivery)
)

print(seller_train_profile.isna().sum())

seller_id                          0
seller_train_order_count           0
seller_train_item_count            0
seller_train_order_value           0
seller_train_mean_delivery_days    0
seller_train_std_delivery_days     0
seller_train_late_rate             0
seller_train_delivery_smoothed     0
dtype: int64


In [ ]:
# Apply seller profile to validation and test sets
def apply_train_seller_profile(data, seller_profile):

    out = data.merge(
        seller_profile,
        on="seller_id",
        how="left",
        validate="many_to_one"
    )

    # Seller did not appear in training set
    out["new_seller_flag"] = (
        out["seller_train_order_count"].isna()
    ).astype(int)

    # Historical activity
    out["seller_hist_order_count"] = (
        out["seller_train_order_count"]
        .fillna(0)
    )

    out["seller_hist_item_count"] = (
        out["seller_train_item_count"]
        .fillna(0)
    )

    out["seller_hist_order_value"] = (
        out["seller_train_order_value"]
        .fillna(0)
    )

    # Historical delivery performance
    out["seller_hist_delivery_smoothed"] = (
        out["seller_train_delivery_smoothed"]
        .fillna(global_mean_delivery)
    )

    out["seller_hist_std_delivery_days"] = (
        out["seller_train_std_delivery_days"]
        .fillna(global_std_delivery)
    )

    out["seller_hist_late_rate"] = (
        out["seller_train_late_rate"]
        .fillna(global_late_rate)
    )

    return out

In [91]:
val_so = apply_train_seller_profile(
    val_so,
    seller_train_profile
)

test_so = apply_train_seller_profile(
    test_so,
    seller_train_profile
)

## 6. Aggregate seller features to order dataset

In [93]:
def aggregate_seller_features(data):

    result = (
        data
        .groupby("order_id")
        .agg(
            # Seller experience
            mean_seller_hist_order_count=(
                "seller_hist_order_count",
                "mean"
            ),

            max_seller_hist_order_count=(
                "seller_hist_order_count",
                "max"
            ),

            mean_seller_hist_item_count=(
                "seller_hist_item_count",
                "mean"
            ),

            mean_seller_hist_order_value=(
                "seller_hist_order_value",
                "mean"
            ),

            # Historical delivery performance
            mean_seller_hist_delivery_days=(
                "seller_hist_delivery_smoothed",
                "mean"
            ),

            max_seller_hist_delivery_days=(
                "seller_hist_delivery_smoothed",
                "max"
            ),

            # Delivery stability
            mean_seller_hist_delivery_std=(
                "seller_hist_std_delivery_days",
                "mean"
            ),

            # Historical late performance
            mean_seller_hist_late_rate=(
                "seller_hist_late_rate",
                "mean"
            ),

            max_seller_hist_late_rate=(
                "seller_hist_late_rate",
                "max"
            ),

            # New seller information
            new_seller_share=(
                "new_seller_flag",
                "mean"
            )
        )
        .reset_index()
    )

    # 是否至少包含一个新 seller
    result["any_new_seller_flag"] = (
        result["new_seller_share"] > 0
    ).astype(int)

    return result

In [95]:
train_seller_features = aggregate_seller_features(train_so)
val_seller_features = aggregate_seller_features(val_so)
test_seller_features = aggregate_seller_features(test_so)

seller_features = pd.concat(
    [
        train_seller_features,
        val_seller_features,
        test_seller_features
    ],
    ignore_index=True
)

print(seller_features.shape)
print(seller_features.columns.tolist())

display(seller_features.head())

# Merge seller features back to main dataset
df_fe = df.merge(
    seller_features,
    on="order_id",
    how="left",
    validate="one_to_one"
)

print("Before:", df.shape)
print("After:", df_fe.shape)
print(df_fe.columns.tolist())

(96454, 12)
['order_id', 'mean_seller_hist_order_count', 'max_seller_hist_order_count', 'mean_seller_hist_item_count', 'mean_seller_hist_order_value', 'mean_seller_hist_delivery_days', 'max_seller_hist_delivery_days', 'mean_seller_hist_delivery_std', 'mean_seller_hist_late_rate', 'max_seller_hist_late_rate', 'new_seller_share', 'any_new_seller_flag']


,order_id,mean_seller_hist_order_count,max_seller_hist_order_count,mean_seller_hist_item_count,mean_seller_hist_order_value,mean_seller_hist_delivery_days,max_seller_hist_delivery_days,mean_seller_hist_delivery_std,mean_seller_hist_late_rate,max_seller_hist_late_rate,new_seller_share,any_new_seller_flag
0,00010242fe8c5a6d1ba2dd792cb16214,83.0,83.0,96.0,7446.4,13.583843,13.583843,11.913763,0.048193,0.048193,0.0,0
1,00018f77f2f0320c557190d7a144bdd3,0.0,0.0,0.0,0.0,13.139497,13.139497,9.833532,0.076656,0.076656,1.0,1
2,000229ec398224ef6ca0657da4fc703e,6.0,6.0,7.0,1543.0,13.526936,13.526936,5.934830,0.000000,0.000000,0.0,0
3,00042b26cf59d7ce69dfabb4e55b4fd9,3.0,3.0,3.0,156.9,12.789013,12.789013,4.960795,0.000000,0.000000,0.0,0
4,00048cc3ae777c65dbb7d2a0634bc1ea,0.0,0.0,0.0,0.0,13.139497,13.139497,9.833532,0.076656,0.076656,1.0,1


Before: (96454, 59)
After: (96454, 70)
['order_id', 'order_purchase_timestamp', 'purchase_month', 'customer_state', 'customer_region', 'customer_zip_code_prefix', 'customer_zip_lat', 'customer_zip_lng', 'customer_zip_density', 'customer_is_capital', 'remote_flag', 'order_value', 'mean_item_price', 'max_item_price', 'freight_value', 'item_count', 'unique_product_count', 'seller_count', 'seller_state_count', 'product_category_count', 'multi_seller_flag', 'product_category', 'primary_seller_state', 'primary_seller_region', 'region_pair', 'same_state', 'any_seller_same_state', 'same_city', 'any_seller_same_city', 'mean_seller_zip_lat', 'mean_seller_zip_lng', 'estimated_min_distance_km', 'estimated_mean_distance_km', 'estimated_max_distance_km', 'geolocation_missing', 'total_weight_g', 'total_volume_cm3', 'max_item_weight_g', 'longest_side_cm', 'max_sum_of_dims_cm', 'volumetric_weight_kg', 'billable_weight_kg', 'density_g_cm3', 'payment_value', 'payment_installments', 'payment_type_count', 

In [96]:
output_path = r"D:\NUS Note\IT5006\it5006_project\data\preprocessed_data\df_fe_b.csv"

df_fe.to_csv(
    output_path,
    index=False
)

print(f"Saved to: {output_path}")

Saved to: D:\NUS Note\IT5006\it5006_project\data\preprocessed_data\df_fe_b.csv


## 5. Encoding and scaling inside sklearn pipelines

In [92]:
# TODO